In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("lazy-eval-lab") \
    .getOrCreate()

In [3]:
raw = spark.read.parquet("/opt/spark-apps/data/cleaned/prices")

no_shuffle = raw.filter(F.col("year") == 2021)
no_shuffle.explain()

with_shuffle = raw.groupBy("stock_code").count()
with_shuffle.explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [stock_code#0,bas_dt#1,close_price#2L,fluctuation_rate#3,listed_share_count#4L,market_cap#5L,open_price#6L,snapshot_type#7,is_interpolated#8,split_suspected#9,year#10] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/spark-apps/data/cleaned/prices], PartitionFilters: [isnotnull(year#10), (year#10 = 2021)], PushedFilters: [], ReadSchema: struct<stock_code:string,bas_dt:string,close_price:bigint,fluctuation_rate:double,listed_share_co...


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[stock_code#0], functions=[count(1)])
   +- Exchange hashpartitioning(stock_code#0, 200), ENSURE_REQUIREMENTS, [plan_id=20]
      +- HashAggregate(keys=[stock_code#0], functions=[partial_count(1)])
         +- Project [stock_code#0]
            +- FileScan parquet [stock_code#0,year#10] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[

In [4]:
grouped = raw.groupBy("stock_code").count()
print("현재 shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("실제 결과 파티션 수:", grouped.rdd.getNumPartitions())

spark.conf.set("spark.sql.shuffle.partitions", "4")

grouped2 = raw.groupBy("stock_code").count()
print("변경 후 shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("실제 결과 파티션 수:", grouped2.rdd.getNumPartitions())

현재 shuffle.partitions: 200
실제 결과 파티션 수: 1
변경 후 shuffle.partitions: 4
실제 결과 파티션 수: 1


In [5]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

grouped3 = raw.groupBy("stock_code").count()
print("AQE 끈 상태, shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("실제 결과 파티션 수:", grouped3.rdd.getNumPartitions())

AQE 끈 상태, shuffle.partitions: 4
실제 결과 파티션 수: 4


In [6]:
indicators = spark.read.parquet("/opt/spark-apps/data/indicators")
companies = spark.read.parquet("/opt/spark-apps/data/raw/companies").filter(F.col("corp_cls") == "Y")
market_codes = companies.select("stock_code").distinct()

joined = indicators.join(F.broadcast(market_codes), on="stock_code", how="inner")
joined.explain()

== Physical Plan ==
*(3) Project [stock_code#91, fs_div#92, currency#93, rcept_no#94, eps#95, bps#96, per#97, pbr#98, roe#99, debt_ratio#100, dividend_yield#101, roa#102, momentum#103, f_score#104, eps_growth_rate#105, year#106]
+- *(3) BroadcastHashJoin [stock_code#91], [stock_code#126], Inner, BuildRight, false
   :- *(3) Filter isnotnull(stock_code#91)
   :  +- *(3) ColumnarToRow
   :     +- FileScan parquet [stock_code#91,fs_div#92,currency#93,rcept_no#94,eps#95,bps#96,per#97,pbr#98,roe#99,debt_ratio#100,dividend_yield#101,roa#102,momentum#103,f_score#104,eps_growth_rate#105,year#106] Batched: true, DataFilters: [isnotnull(stock_code#91)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/opt/spark-apps/data/indicators], PartitionFilters: [], PushedFilters: [IsNotNull(stock_code)], ReadSchema: struct<stock_code:string,fs_div:string,currency:string,rcept_no:string,eps:double,bps:double,per:...
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true])

In [ ]:
bbb